## Phase 1: Library Imports and Data Cleaning
Load the dataset and transform the list of strings into a One-Hot matrix for the Apriori algorithm.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')

CLUSTER_NAMES_MAP = {
    0: "The Plant-Based Household",
    1: "The Family Budget Optimizer",
    2: "The Promotion-Driven Pet Parent",
    3: "The Demanding Premium Tech Consumer",
    4: "The Wellness Customer",
    5: "The Affluent Health-Conscious Buyer"
}


def load_and_prepare_data(basket_path='customer_basket (1).csv', clusters_path='dataset_clusters.csv'):
    basket = pd.read_csv(basket_path)
    clusters = pd.read_csv(clusters_path)
    clusters['cluster'] = clusters['cluster'].map(CLUSTER_NAMES_MAP)
    return pd.merge(clusters, basket, on='customer_id', how='inner')


def convert_string_to_list(string):
    return json.loads(string.replace("'", '"'))


def calculate_metric_range(rules_df, metric='lift'):
    if rules_df.empty:
        return f"{metric} Range: N/A (No rules generated)"
    min_val = rules_df[metric].min()
    max_val = rules_df[metric].max()
    return f"{metric} Range: {max_val - min_val:.4f} (Min: {min_val:.4f}, Max: {max_val:.4f})"


def plot_top_products(basket_list, title_suffix=""):
    te = TransactionEncoder()
    te_ary = te.fit(basket_list).transform(basket_list)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    top_items = df_trans.sum().sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(10, 5))
    top_items.plot(kind='bar', color='#5c6bc0', edgecolor='black')
    plt.title(f'Top 10 Most Frequent Products - {title_suffix}', fontweight='bold')
    plt.ylabel('Absolute Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    filename = f"top_products_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.close()


def plot_association_results(rules_df, title_suffix=""):
    if rules_df.empty:
        return
    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(rules_df['support'], rules_df['confidence'], 
                          c=rules_df['lift'], cmap='viridis', alpha=0.6)
    plt.colorbar(scatter, label='Lift')
    plt.title(f'Rules Scatter Plot (Support vs Confidence) - {title_suffix}', fontweight='bold')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    filename = f"rules_scatter_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.close()


def run_association_rules(basket_data, min_support=0.05, min_confidence=0.2, split_train=True):
    if split_train:
        train_size = int(len(basket_data) * 0.8)
        data_to_model = basket_data[:train_size]
    else:
        data_to_model = basket_data
        
    te = TransactionEncoder()
    te_fit = te.fit(data_to_model).transform(data_to_model)
    transactions_items = pd.DataFrame(te_fit, columns=te.columns_)
    
    frequent_itemsets = apriori(transactions_items, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        return frequent_itemsets, pd.DataFrame()
        
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)
    return frequent_itemsets, rules


if __name__ == "__main__":
    try:
        df_basket_clusters = load_and_prepare_data(
            basket_path='customer_basket (1).csv', 
            clusters_path='dataset_clusters.csv'
        )
        df_basket_clusters['list_of_goods_parsed'] = df_basket_clusters['list_of_goods'].apply(convert_string_to_list)
        
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        pd.set_option('display.max_colwidth', None)

        def clean_rules_df(df_rules):
            if df_rules.empty:
                return df_rules
            df_cool = df_rules.copy()
            df_cool['antecedents'] = df_cool['antecedents'].apply(lambda x: f"[{', '.join(list(x))}]")
            df_cool['consequents'] = df_cool['consequents'].apply(lambda x: f"[{', '.join(list(x))}]")
            metrics = ['support', 'confidence', 'lift', 'leverage', 'conviction', 'zhangs_metric']
            for m in metrics:
                if m in df_cool.columns:
                    df_cool[m] = df_cool[m].round(4)
            return df_cool

        def clean_items_df(df_items):
            if df_items.empty:
                return df_items
            df_cool = df_items.copy()
            df_cool['itemsets'] = df_cool['itemsets'].apply(lambda x: f"[{', '.join(list(x))}]")
            df_cool['support'] = df_cool['support'].round(4)
            return df_cool

        
        # PART 1: GLOBAL RULES (ALL CUSTOMERS)

        print("\n" + "="*80)
        print("=== PART 1: GLOBAL ASSOCIATION RULES (ALL CUSTOMERS COMBINED) ===")
        print("="*80)
        all_baskets = df_basket_clusters['list_of_goods_parsed'].tolist()
        plot_top_products(all_baskets, "Global Dataset")
        
        items_all, rules_all = run_association_rules(all_baskets, min_support=0.02, min_confidence=0.15, split_train=True)
        
        print(f"\n-> [Global] Frequent Itemsets (Top 10):")
        display(clean_items_df(items_all.head(10)))
        
        print(f"\n-> [Global] Association Rules Generated (Top 15 sorted by Lift):")
        if not rules_all.empty:
            df_all_view = rules_all[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)
            display(clean_rules_df(df_all_view))
            print(f"\n{calculate_metric_range(rules_all, 'lift')}")
            plot_association_results(rules_all, "Global Dataset")
        else:
            print("No rules generated globally for the current threshold limits.")

        # PART 2: STANDARDIZED ANALYSIS PER CLUSTER (AUTOMATED LOOP)

        print("\n" + "="*80)
        print("=== PART 2: STANDARDIZED ASSOCIATION RULES ANALYSIS BY CLUSTER COEFFICIENT ===")
        print("="*80)
        
        DEFAULT_SUPPORT = 0.02
        DEFAULT_CONFIDENCE = 0.15
        
        for cluster_id, cluster_name in CLUSTER_NAMES_MAP.items():
            print("\n" + "-"*60)
            print(f"ANALYSIS FOR CLUSTER {cluster_id}: {cluster_name}")
            print("-"*60)
            
            df_sub_cluster = df_basket_clusters[df_basket_clusters['cluster'] == cluster_name]
            cluster_baskets = df_sub_cluster['list_of_goods_parsed'].tolist()
            
            print(f"Total transactions within this segment: {len(cluster_baskets)}")
            
            if len(cluster_baskets) > 0:
                plot_top_products(cluster_baskets, cluster_name)
                
                items_cl, rules_cl = run_association_rules(
                    cluster_baskets, 
                    min_support=DEFAULT_SUPPORT, 
                    min_confidence=DEFAULT_CONFIDENCE, 
                    split_train=True
                )
                
                print(f"\n   -> Frequent Itemsets Found (Top 10 Sample out of {len(items_cl)}):")
                if not items_cl.empty:
                    display(clean_items_df(items_cl.head(10)))
                else:
                    print(f"      No frequent itemsets passed the {DEFAULT_SUPPORT*100}% support threshold.")
                
                print(f"\n   -> Association Rules Found (Top 10 sorted by Lift out of {len(rules_cl)}):")
                if not rules_cl.empty:
                    df_cl_view = rules_cl[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)
                    display(clean_rules_df(df_cl_view))
                    print(f"\n      {calculate_metric_range(rules_cl, 'lift')}")
                    plot_association_results(rules_cl, cluster_name)
                else:
                    print(f"      No rules generated for the thresholds (Support={DEFAULT_SUPPORT*100}%, Confidence={DEFAULT_CONFIDENCE*100}%).\n"
                          f"      Indicates independent purchasing behavior for products within this segment.")
            else:
                print(f"   -> Alert: No records found in the dataset for this cluster.")

        print("\n" + "="*80)
        print("=== DATA PROCESSING COMPLETED SUCCESSFULLY ===")
        print("="*80)

    except Exception as e:
        print(f"\n[Execution Error]: {e}")


=== PART 1: GLOBAL ASSOCIATION RULES (ALL CUSTOMERS COMBINED) ===

-> [Global] Frequent Itemsets (Top 10):


,support,itemsets
0,0.1203,[airpods]
1,0.0632,[almonds]
2,0.0698,[antioxydant juice]
3,0.1274,[asparagus]
4,0.0664,[avocado]
5,0.0844,[babies food]
6,0.0688,[bacon]
7,0.0510,[barbecue sauce]
8,0.0558,[beer]
9,0.0452,[black beer]



-> [Global] Association Rules Generated (Top 15 sorted by Lift):


,antecedents,consequents,support,confidence,lift
0,[energy drink],[airpods],0.0248,0.3372,2.8029
1,[airpods],[energy drink],0.0248,0.2058,2.8029
2,[bluetooth headphones],[airpods],0.0241,0.3355,2.7887
3,[airpods],[bluetooth headphones],0.0241,0.2000,2.7887
4,[cereals],[eggs],0.0224,0.2213,2.3772
5,[eggs],[cereals],0.0224,0.2408,2.3772
6,[butter],[eggs],0.0206,0.2136,2.2941
7,[eggs],[butter],0.0206,0.2216,2.2941
8,[fresh bread],[cereals],0.0228,0.2286,2.2568
9,[cereals],[fresh bread],0.0228,0.2246,2.2568



lift Range: 0.7120 (Min: 2.0909, Max: 2.8029)

=== PART 2: STANDARDIZED ASSOCIATION RULES ANALYSIS BY CLUSTER COEFFICIENT ===

------------------------------------------------------------
ANALYSIS FOR CLUSTER 0: The Plant-Based Household
------------------------------------------------------------
Total transactions within this segment: 10080

   -> Frequent Itemsets Found (Top 10 Sample out of 271):


...


   -> Association Rules Found (Top 10 sorted by Lift out of 0):
      No rules generated for the thresholds (Support=2.0%, Confidence=15.0%).
      Indicates independent purchasing behavior for products within this segment.

------------------------------------------------------------
ANALYSIS FOR CLUSTER 5: The Affluent Health-Conscious Buyer
------------------------------------------------------------
Total transactions within this segment: 4832

   -> Frequent Itemsets Found (Top 10 Sample out of 267):


,support,itemsets
0,0.0655,[airpods]
1,0.0461,[almonds]
2,0.0528,[antioxydant juice]
3,0.0893,[asparagus]
4,0.0422,[avocado]
5,0.2435,[babies food]
6,0.0411,[bacon]
7,0.0507,[barbecue sauce]
8,0.0507,[beer]
9,0.0463,[black beer]



   -> Association Rules Found (Top 10 sorted by Lift out of 164):


,antecedents,consequents,support,confidence,lift
0,"[babies food, napkins]",[cooking oil],0.0318,0.4155,1.8439
1,"[babies food, cooking oil]",[napkins],0.0318,0.3880,1.7120
2,"[dog food, cooking oil]",[babies food],0.0298,0.4137,1.6991
3,"[cooking oil, napkins]",[babies food],0.0318,0.4128,1.6953
4,"[dog food, babies food]",[cooking oil],0.0298,0.3734,1.6568
5,"[dog food, cooking oil]",[napkins],0.0261,0.3633,1.6030
6,"[dog food, napkins]",[cooking oil],0.0261,0.3495,1.5508
7,"[babies food, cooking oil]",[dog food],0.0298,0.3628,1.5459
8,[milk],[napkins],0.0492,0.3442,1.5187
9,[napkins],[milk],0.0492,0.2169,1.5187



      lift Range: 0.8252 (Min: 1.0188, Max: 1.8439)

=== DATA PROCESSING COMPLETED SUCCESSFULLY ===


## Main Marketing Insights & Actions

### Insight A: Global Store Drivers (Traffic Magnets)
* **Finding:** Across the entire store, items like `asparagus`, `airpods`, and `cereals` dominate absolute purchasing frequencies. 
* **Marketing Action:** Treat these as **"Loss-Leaders"** in mass-media campaigns or catalog covers. High-reach discounts on these specific items will efficiently drive overall traffic to the checkout.

### Insight B: Cluster 4 — "The Wellness Customer" (High-Conversion Target)
* **Finding:** This segment yielded the most robust patterns, generating **14 strong association rules** with a **Maximum Lift of 2.87**. Customers are nearly 3 times more likely to buy a secondary healthy item if promoted next to the cluster's anchor item (`asparagus`).
* **Marketing Action — "Live Well" Campaign:** Launch immediate product bundles or targeted discount coupons pairing `asparagus` with its linked wellness items. Trigger automated "Frequently Bought Together" carousels at the digital checkout for this cluster.

### Insight C: Cluster 3 — "The Demanding Premium Tech Consumer" (Anti-Bundling Strategy)
* **Finding:** Cluster 3 generated **no meaningful association rules**. Purchases within this segment are statistically independent (buying Tech Product A does not influence buying Tech Product B).
* **Marketing Action — Individual Margin Strategy:** **Do not waste budget** creating product bundles or multi-pack discounts for this group. Pivot exclusively to **Up-Selling** (premium upgrades of the same item) or brand-loyalty perks (e.g., Apple- or Samsung-specific campaigns).


In [4]:
print(load_and_prepare_data(basket_path='customer_basket (1).csv', clusters_path='dataset_clusters.csv')
)

       customer_id          customer_name  number_complaints  distinct_stores_visited  lifetime_spend_groceries  lifetime_spend_electronics  typical_hour  lifetime_spend_vegetables  lifetime_spend_nonalcohol_drinks  lifetime_spend_alcohol_drinks  lifetime_spend_meat  lifetime_spend_fish  lifetime_spend_hygiene  lifetime_spend_videogames  lifetime_spend_petfood  lifetime_total_distinct_products  percentage_of_products_bought_promotion  year_first_transaction   latitude  longitude   age  female  total_kids  total_lifetime_spend  loyalty_product_type  has_loyalty_card  lifetime_spend_groceries_prop  lifetime_spend_electronics_prop  lifetime_spend_vegetables_prop  lifetime_spend_nonalcohol_drinks_prop  lifetime_spend_alcohol_drinks_prop  lifetime_spend_meat_prop  lifetime_spend_fish_prop  lifetime_spend_hygiene_prop  lifetime_spend_videogames_prop  lifetime_spend_petfood_prop                          cluster  invoice_id  \
0                3  Bsc. Crystal Kitchens                1.0       

In [6]:
import pandas as pd
import json
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

class BasketAnalyzer:
    def __init__(self, basket_path: str, clusters_path: str):
        # Carregamento e preparação idênticos
        self.df = self._prepare_data(basket_path, clusters_path)

    def _prepare_data(self, basket_path: str, clusters_path: str) -> pd.DataFrame:
        basket = pd.read_csv(basket_path)
        clusters = pd.read_csv(clusters_path)
        
        # Mapeamento de clusters (conforme o seu CLUSTER_NAMES_MAP)
        cluster_map = {
            0: "The Plant-Based Household", 1: "The Family Budget Optimizer",
            2: "The Promotion-Driven Pet Parent", 3: "The Demanding Premium Tech Consumer",
            4: "The Wellness Customer", 5: "The Affluent Health-Conscious Buyer"
        }
        clusters['cluster'] = clusters['cluster'].map(cluster_map)
        
        df = clusters.merge(basket, on='customer_id')
        df['list_of_goods_parsed'] = df['list_of_goods'].apply(lambda x: json.loads(x.replace("'", '"')))
        return df

    def _clean_rules(self, df: pd.DataFrame) -> pd.DataFrame:
        if df.empty: return df
        df_c = df.copy()
        # Formatação idêntica: lista com strings dentro de colchetes
        df_c['antecedents'] = df_c['antecedents'].apply(lambda x: f"[{', '.join(list(x))}]")
        df_c['consequents'] = df_c['consequents'].apply(lambda x: f"[{', '.join(list(x))}]")
        return df_c.round(4)

    def _clean_items(self, df: pd.DataFrame) -> pd.DataFrame:
        if df.empty: return df
        df_c = df.copy()
        df_c['itemsets'] = df_c['itemsets'].apply(lambda x: f"[{', '.join(list(x))}]")
        return df_c.round(4)

    def get_analysis(self, cluster_name: str = None, min_sup: float = 0.02, min_conf: float = 0.15):
        """Retorna o dataset filtrado, itemsets e regras limpas."""
        target_df = self.df if cluster_name is None else self.df[self.df['cluster'] == cluster_name]
        data = target_df['list_of_goods_parsed'].tolist()
        
        te = TransactionEncoder()
        df_trans = pd.DataFrame(te.fit(data).transform(data), columns=te.columns_)
        
        itemsets = apriori(df_trans, min_support=min_sup, use_colnames=True)
        rules = association_rules(itemsets, metric="confidence", min_threshold=min_conf)
        
        return self._clean_items(itemsets.head(10)), self._clean_rules(rules.sort_values('lift', ascending=False))

# --- Execução Equivalente ---
if __name__ == "__main__":
    analyzer = BasketAnalyzer('customer_basket (1).csv', 'dataset_clusters.csv')
    
    # 1. Global
    items, rules = analyzer.get_analysis()
    print("=== GLOBAL ASSOCIATION RULES ===")
    display(items)
    display(rules.head(15))
    
    # 2. Por Cluster
    for cluster in analyzer.df['cluster'].dropna().unique():
        print(f"\n--- Cluster: {cluster} ---")
        items, rules = analyzer.get_analysis(cluster_name=cluster)
        display(items.head(10))
        display(rules.head(10))

=== GLOBAL ASSOCIATION RULES ===


,support,itemsets
0,0.1214,[airpods]
1,0.0641,[almonds]
2,0.0693,[antioxydant juice]
3,0.1281,[asparagus]
4,0.0666,[avocado]
5,0.0832,[babies food]
6,0.0682,[bacon]
7,0.0512,[barbecue sauce]
8,0.0553,[beer]
9,0.0455,[black beer]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,[bluetooth headphones],[airpods],0.0728,0.1214,0.0248,0.3414,2.8110,1.0,0.0160,1.3340,0.6948,0.1467,0.2503,0.2730
1,[airpods],[bluetooth headphones],0.1214,0.0728,0.0248,0.2046,2.8110,1.0,0.0160,1.1657,0.7333,0.1467,0.1422,0.2730
2,[energy drink],[airpods],0.0752,0.1214,0.0253,0.3355,2.7629,1.0,0.0161,1.3222,0.6900,0.1473,0.2437,0.2717
3,[airpods],[energy drink],0.1214,0.0752,0.0253,0.2079,2.7629,1.0,0.0161,1.1675,0.7263,0.1473,0.1434,0.2717
10,[cereals],[eggs],0.0995,0.0924,0.0218,0.2191,2.3704,1.0,0.0126,1.1622,0.6420,0.1281,0.1395,0.2275
11,[eggs],[cereals],0.0924,0.0995,0.0218,0.2359,2.3704,1.0,0.0126,1.1785,0.6370,0.1281,0.1515,0.2275
13,[cereals],[fresh bread],0.0995,0.0993,0.0225,0.2260,2.2749,1.0,0.0126,1.1636,0.6223,0.1275,0.1406,0.2262
12,[fresh bread],[cereals],0.0993,0.0995,0.0225,0.2264,2.2749,1.0,0.0126,1.1640,0.6222,0.1275,0.1409,0.2262
6,[butter],[eggs],0.0965,0.0924,0.0202,0.2087,2.2586,1.0,0.0112,1.1470,0.6168,0.1194,0.1282,0.2134
7,[eggs],[butter],0.0924,0.0965,0.0202,0.2180,2.2586,1.0,0.0112,1.1554,0.6140,0.1194,0.1345,0.2134



--- Cluster: The Family Budget Optimizer ---


,support,itemsets
0,0.1338,[airpods]
1,0.0577,[almonds]
2,0.0595,[antioxydant juice]
3,0.1611,[asparagus]
4,0.0822,[avocado]
5,0.0555,[babies food]
6,0.0859,[bacon]
7,0.0515,[barbecue sauce]
8,0.0575,[beer]
9,0.0407,[black beer]


...

,support,itemsets
0,0.1660,[airpods]
1,0.0600,[almonds]
2,0.0398,[antioxydant juice]
3,0.1195,[asparagus]
4,0.0500,[avocado]
5,0.0540,[babies food]
6,0.0573,[bacon]
7,0.0565,[barbecue sauce]
8,0.0739,[beer]
9,0.0507,[black beer]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
11,[bluetooth headphones],[energy drink],0.0899,0.0896,0.0226,0.2512,2.8022,1.0,0.0145,1.2158,0.7067,0.1439,0.1775,0.2516
12,[energy drink],[bluetooth headphones],0.0896,0.0899,0.0226,0.2520,2.8022,1.0,0.0145,1.2167,0.7065,0.1439,0.1781,0.2516
0,[bluetooth headphones],[airpods],0.0899,0.1660,0.0372,0.4138,2.4936,1.0,0.0223,1.4229,0.6582,0.1702,0.2972,0.3191
1,[airpods],[bluetooth headphones],0.1660,0.0899,0.0372,0.2243,2.4936,1.0,0.0223,1.1732,0.7182,0.1702,0.1476,0.3191
4,[energy drink],[airpods],0.0896,0.1660,0.0356,0.3974,2.3945,1.0,0.0207,1.3841,0.6397,0.1619,0.2775,0.3060
5,[airpods],[energy drink],0.1660,0.0896,0.0356,0.2147,2.3945,1.0,0.0207,1.1592,0.6983,0.1619,0.1373,0.3060
9,[pancakes],[airpods],0.0689,0.1660,0.0216,0.3130,1.8861,1.0,0.0101,1.2141,0.5046,0.1012,0.1763,0.2215
3,[energy bar],[airpods],0.0772,0.1660,0.0236,0.3058,1.8426,1.0,0.0108,1.2015,0.4956,0.1075,0.1677,0.2240
6,[iphone 10],[airpods],0.0755,0.1660,0.0222,0.2937,1.7694,1.0,0.0096,1.1808,0.4703,0.1011,0.1531,0.2136
7,[laptop],[airpods],0.0904,0.1660,0.0258,0.2853,1.7188,1.0,0.0108,1.1669,0.4597,0.1118,0.1430,0.2203



--- Cluster: The Affluent Health-Conscious Buyer ---


,support,itemsets
0,0.0675,[airpods]
1,0.0482,[almonds]
2,0.0526,[antioxydant juice]
3,0.0884,[asparagus]
4,0.0435,[avocado]
5,0.2374,[babies food]
6,0.0428,[bacon]
7,0.0499,[barbecue sauce]
8,0.0509,[beer]
9,0.0466,[black beer]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
153,"[babies food, napkins]",[cooking oil],0.0768,0.2266,0.0308,0.4016,1.7723,1.0,0.0134,1.2925,0.4720,0.1131,0.2263,0.2688
151,"[dog food, cooking oil]",[babies food],0.0745,0.2374,0.0302,0.4056,1.7085,1.0,0.0125,1.2829,0.4481,0.1073,0.2205,0.2664
155,"[babies food, cooking oil]",[napkins],0.0797,0.2274,0.0308,0.3870,1.7016,1.0,0.0127,1.2603,0.4480,0.1116,0.2065,0.2613
154,"[cooking oil, napkins]",[babies food],0.0776,0.2374,0.0308,0.3973,1.6739,1.0,0.0124,1.2654,0.4364,0.1085,0.2097,0.2636
150,"[dog food, babies food]",[cooking oil],0.0809,0.2266,0.0302,0.3734,1.6477,1.0,0.0119,1.2343,0.4277,0.1090,0.1898,0.2534
159,"[dog food, cooking oil]",[napkins],0.0745,0.2274,0.0275,0.3694,1.6243,1.0,0.0106,1.2252,0.4153,0.1003,0.1838,0.2452
152,"[babies food, cooking oil]",[dog food],0.0797,0.2382,0.0302,0.3792,1.5920,1.0,0.0112,1.2272,0.4041,0.1050,0.1851,0.2530
160,"[dog food, napkins]",[cooking oil],0.0784,0.2266,0.0275,0.3509,1.5486,1.0,0.0098,1.1915,0.3844,0.0992,0.1607,0.2362
134,[napkins],[milk],0.2274,0.1407,0.0482,0.2120,1.5065,1.0,0.0162,1.0905,0.4352,0.1507,0.0830,0.2773
133,[milk],[napkins],0.1407,0.2274,0.0482,0.3426,1.5065,1.0,0.0162,1.1753,0.3913,0.1507,0.1491,0.2773
